# Part 4: Deep Security Threat Analysis

This section identifies complex attack patterns, analyzes suspicious behavior, and assigns risk levels to network entities.

(a) Detecting Complex Attack Patterns

Port Scan Detection

In [75]:
port_scan = df.groupby(['source', 'time_window'])['destinationPort'].nunique()

port_scanners = port_scan[port_scan > 20]

port_scanners.head()

source         time_window        
192.168.1.101  2010-06-17 13:00:00      22
192.168.1.105  2010-06-13 16:00:00    1014
               2010-06-17 10:00:00      24
192.168.2.107  2010-06-14 21:00:00     717
               2010-06-14 22:00:00    2258
Name: destinationPort, dtype: int64

A port scan is identified when a single source IP contacts multiple destination ports within a short time window, indicating probing behavior.

Slow DDoS Detection

In [76]:
ddos = df.groupby(['destination', 'time_window']).size()

rolling = ddos.groupby(level=0).rolling(3).mean()

slow_ddos = rolling[rolling > rolling.mean() + 2 * rolling.std()]

slow_ddos.head()

destination    destination    time_window        
115.85.145.2   115.85.145.2   2010-06-15 12:00:00    610.666667
                              2010-06-15 13:00:00    643.666667
                              2010-06-15 14:00:00    545.666667
                              2010-06-15 20:00:00    513.666667
123.50.56.175  123.50.56.175  2010-06-15 11:00:00    824.333333
dtype: float64

Slow DDoS attacks are detected using rolling averages to identify sustained abnormal traffic rather than sudden spikes.

IP Hopping Detection

In [77]:
behavior = df.groupby('source')[['destination', 'destinationPort']].nunique()

ip_hopping = behavior[
    (behavior['destination'] > 50) &
    (behavior['destinationPort'] > 50)
]

ip_hopping.head()

,destination,destinationPort
source,,
192.168.1.101,2128,235
192.168.1.102,1521,130
192.168.1.103,1842,202
192.168.1.104,1448,396
192.168.1.105,1979,1610


IP hopping behavior is characterized by a single entity communicating with many destinations and ports, potentially to evade detection.

(b) Malicious Payload Analysis

Suspicious Payloads

In [78]:
keywords = ['cmd', 'exec', 'shell', 'attack', 'malware']

def suspicious(x):
    if pd.isna(x):
        return False
    return any(k in str(x).lower() for k in keywords)

df['suspicious_payload'] = df['sourcePayloadAsUTF'].apply(suspicious)

suspicious_flows = df[df['suspicious_payload']]

len(suspicious_flows)

1184

Payload inspection is used to detect potentially malicious commands or exploit patterns.

Encrypted Traffic Anomalies

In [84]:
df['payload_length'] = df['sourcePayloadAsUTF'].astype(str).apply(len)

large_payloads = df[
    df['payload_length'] > df['payload_length'].quantile(0.99)
]

len(large_payloads)

8207

Payload inspection is used to detect potentially malicious commands or exploit patterns.

Since payload fields are sparse, anomaly detection is performed using payload size instead of keyword matching. Extremely large payloads may indicate encoded or exfiltrated data.

C2 Detection

In [81]:
c2_candidates = df.groupby(['source', 'destination']).size()

c2_suspects = c2_candidates[c2_candidates > 100]

c2_suspects.head()

,,0
source,destination,
0.0.0.0,0.0.0.0,1553
131.202.240.209,192.168.5.122,1067
131.202.240.218,192.168.5.122,2023
131.202.243.90,192.168.5.122,5221
142.167.88.44,192.168.5.122,452


Repeated communication between the same source and destination may indicate command-and-control activity.

(c) Risk Classification (VERY IMPORTANT)

Risk Scoring

In [82]:
risk_table = []

for ip in df['source'].unique()[:1000]:  # sample for speed
    score = 0

    if ip in port_scanners.index.get_level_values(0):
        score += 2
    if ip in scanners.index.get_level_values(0):
        score += 2
    if ip in multi_proto_suspects.index.get_level_values(0):
        score += 1

    if score >= 4:
        risk = "HIGH"
    elif score >= 2:
        risk = "MEDIUM"
    else:
        risk = "LOW"

    risk_table.append((ip, risk))

risk_df = pd.DataFrame(risk_table, columns=['IP', 'Risk'])

risk_df.head()

,IP,Risk
0,192.168.4.121,HIGH
1,192.168.4.119,MEDIUM
2,192.168.3.116,HIGH
3,192.168.2.113,HIGH
4,192.168.4.118,HIGH


Risk levels are assigned based on multiple indicators of suspicious behavior. High-risk IPs exhibit several attack patterns, while low-risk IPs show minimal anomalies.

SUMMARY

In [83]:
summary = {
    "Total Flows": len(df),
    "Unique IPs": exact_unique_ips,
    "Traffic Spikes": len(anomalies),
    "Packet Anomalies": len(packet_anomalies),
    "DDoS Suspects": len(ddos_suspects),
    "Scan Suspects": len(scanners),
    "High Risk IPs": len(risk_df[risk_df['Risk']=="HIGH"])
}

summary

{'Total Flows': 2071657,
 'Unique IPs': 34801,
 'Traffic Spikes': 2,
 'Packet Anomalies': 48,
 'DDoS Suspects': 13,
 'Scan Suspects': 847,
 'High Risk IPs': 10}

The analysis reveals multiple indicators of anomalous and potentially malicious behavior. Significant traffic spikes align with abnormal protocol usage and behavioral anomalies, suggesting coordinated activity. Several IPs exhibit characteristics of scanning and distributed traffic targeting, indicating possible attack patterns such as port scanning and DDoS. The risk classification framework aggregates these signals to identify high-risk entities for further investigation.

FINAL ANALYSIS

For Port Scan:

The observed behavior strongly indicates port scanning activity, as a single source probes a large number of ports within a short time window.

For DDoS:

The large number of distinct sources targeting a single destination suggests a distributed denial-of-service (DDoS) attack pattern.

For IP Hopping:

Communication with a large number of destinations and ports suggests automated or evasive behavior, potentially indicative of malicious activity.